# Revisión visual del pipeline — samples aleatorios

Cada run de este notebook elige estrellas **distintas** (RNG sin seed, a propósito)
y muestra todos los pasos del pipeline para auditarlos visualmente:

1. LC cruda → limpieza (`clean_lightcurve`, puntos descartados en rojo)
2. Periodogramas LS + ACF con los peaks seleccionados (`msv.peaks`)
3. hist2d (32×32) de cada peak — lo que ve la CNN
4. LC plegada al mejor peak
5. Clase BRF por peak + incertidumbre (`instability`), y qué relabeló el gate

No entrena ni ejecuta la CNN: la parte BRF se lee del CSV precalculado por
`scripts/run_brf_snr.py` (si existe). Corre en el env **CNN_TESS** (sin TF).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from msv import config, viz

# ---- Config -----------------------------------------------------------------
# dataset: "masivas" (default) u "ogle"
DATASET = "masivas"
N_SAMPLES = 3

PATHS = {
    "masivas": dict(
        lc_parquet=config.LC_PARQUET_MASSIVE,
        peaks_path=config.PEAKS_PARQUET,
        ls_path=config.PERIODOGRAMS_LS,
        acf_path=config.PERIODOGRAMS_ACF,
        # CSV del run final (task 2c); None si aún no existe
        brf_csv=config.RESULTS_DIR / "brf_mc_peaks_final.csv",
    ),
    "ogle": dict(
        lc_parquet=config.LC_PARQUET_OGLE,
        peaks_path=config.REPO_ROOT / "peaks_statsmodels.parquet",
        ls_path=config.PERIODOGRAMS_LS,    # no cubre OGLE -> se computa en vivo
        acf_path=config.PERIODOGRAMS_ACF,
        brf_csv=config.RESULTS_DIR / "brf_mc_peaks_Number_ELL.csv",
    ),
}[DATASET]
if PATHS["brf_csv"] is not None and not PATHS["brf_csv"].exists():
    print(f"(sin CSV de BRF en {PATHS['brf_csv']} -> se omite el panel BRF)")
    PATHS["brf_csv"] = None

## Samples aleatorios

Re-ejecutar esta celda (o el notebook completo) muestra estrellas nuevas cada vez.

In [ ]:
pairs = viz.random_pairs(N_SAMPLES, path=PATHS["peaks_path"])
print("Pares elegidos:", pairs)

results = {}
for TIC, sector in pairs:
    print(f"\n{'='*70}\n  TIC {TIC} — sector {sector}\n{'='*70}")
    try:
        results[(TIC, sector)] = viz.show_sample(TIC, sector, **PATHS)
    except ValueError as e:
        print(f"  saltado: {e}")

## Solo estrellas que el gate relabeló

Para auditar qué está **matando** el gate de incertidumbre: samplear únicamente
pares donde algún peak pasó de clase periódica → `Rndm` por `instability`.
Requiere el CSV del run con gate aplicado (`brf_class_raw` presente).

In [ ]:
if PATHS["brf_csv"] is None:
    print("Sin CSV de BRF: correr scripts/run_brf_snr.py primero.")
else:
    brf = pd.read_csv(PATHS["brf_csv"])
    if "brf_class_raw" not in brf.columns:
        print("El CSV no tiene brf_class_raw (gate no aplicado en ese run).")
    else:
        gated = brf[brf["brf_class_raw"] != brf["brf_class"]]
        pares_gated = gated[["TIC", "sector"]].drop_duplicates()
        print(f"{len(pares_gated)} pares con peaks relabelados por el gate")
        import numpy as np
        rng = np.random.default_rng()          # sin seed
        for _, row in pares_gated.sample(min(2, len(pares_gated)),
                                         random_state=None).iterrows():
            viz.show_sample(int(row.TIC), int(row.sector), **PATHS)

## Solo periódicos sobrevivientes

Lo contrario: qué está **dejando pasar** el gate — pares con al menos un peak
clasificado en clase periódica después del gate.

In [ ]:
if PATHS["brf_csv"] is None:
    print("Sin CSV de BRF: correr scripts/run_brf_snr.py primero.")
else:
    brf = pd.read_csv(PATHS["brf_csv"])
    per_ok = brf[brf["brf_class"].isin(config.PERIODIC)]
    pares_per = per_ok[["TIC", "sector"]].drop_duplicates()
    print(f"{len(pares_per)} pares con peaks periódicos tras el gate")
    for _, row in pares_per.sample(min(2, len(pares_per)),
                                   random_state=None).iterrows():
        viz.show_sample(int(row.TIC), int(row.sector), **PATHS)